# Meta Ads Campaign Performance Analysis
## Exploratory Data Analysis (EDA) & Marketing Intelligence
**Technology Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Author:** Data Analyst Portfolio Project  

---
### Objective:
Analyze Facebook and Instagram ad campaign performance across impressions, reach, clicks, conversions, expenditure, and ROAS.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

PROCESSED_DIR = '../data/processed'


### 1. Load Processed Datasets
Loading the cleaned datasets generated in Phase 3.


In [ ]:
campaigns = pd.read_csv(f'{PROCESSED_DIR}/clean_campaign_performance.csv')
ads = pd.read_csv(f'{PROCESSED_DIR}/clean_ad_performance.csv')
daily = pd.read_csv(f'{PROCESSED_DIR}/clean_daily_performance.csv')
users = pd.read_csv(f'{PROCESSED_DIR}/clean_users.csv')
events = pd.read_csv(f'{PROCESSED_DIR}/clean_ad_events.csv')

print(f'Campaigns loaded: {len(campaigns):,}')
print(f'Ads loaded:       {len(ads):,}')
print(f'Users loaded:     {len(users):,}')
print(f'Events loaded:    {len(events):,}')


### 2. Portfolio-Level Executive KPIs
Core metrics calculated across all campaigns and platforms.


In [ ]:
total_spend = campaigns['total_spend'].sum()
total_budget = campaigns['campaign_budget'].sum()
total_imps = campaigns['impressions'].sum()
total_reach = ads['reach'].sum()
total_clicks = campaigns['clicks'].sum()
total_purchases = campaigns['purchases'].sum()
total_revenue = campaigns['revenue'].sum()

kpi_summary = pd.DataFrame({
    'Metric': [
        'Total Budget ($)', 'Total Spend ($)', 'Budget Utilization (%)',
        'Total Impressions', 'Total Reach', 'Total Clicks',
        'Click-Through Rate (CTR %)', 'Cost Per Click (CPC $)', 'Cost Per Mille (CPM $)',
        'Total Purchases (Conversions)', 'Conversion Rate (%)', 'Cost Per Acquisition (CPA $)',
        'Total Revenue ($)', 'Return on Ad Spend (ROAS)'
    ],
    'Value': [
        f'', f'', f'{(total_spend/total_budget)*100:.2f}%',
        f'{total_imps:,}', f'{total_reach:,}', f'{total_clicks:,}',
        f'{(total_clicks/total_imps)*100:.2f}%', f'', f'',
        f'{total_purchases:,}', f'{(total_purchases/total_clicks)*100:.2f}%', f'',
        f'', f'{total_revenue/total_spend:.2f}x'
    ]
})

kpi_summary


### 3. Conversion Funnel Analysis
Visualizing user drop-off from initial impression down to final purchase.


In [ ]:
funnel_data = [
    ('Impressions', (events['event_type'] == 'Impression').sum()),
    ('Clicks', (events['event_type'] == 'Click').sum()),
    ('Engagements', events['event_type'].isin(['Like', 'Comment', 'Share']).sum()),
    ('Purchases', (events['event_type'] == 'Purchase').sum())
]
stages, counts = zip(*funnel_data)
colors = ['#1877F2', '#42B72A', '#F7B125', '#FA383E']

plt.figure(figsize=(10, 5))
bars = plt.barh(stages[::-1], counts[::-1], color=colors[::-1], height=0.55, edgecolor='black')
for bar in bars:
    w = bar.get_width()
    pct = (w / counts[0]) * 100
    plt.text(w + 3000, bar.get_y() + bar.get_height()/2, f'{int(w):,} ({pct:.2f}%)', va='center', fontweight='bold')
plt.title('Meta Ads Full Conversion Funnel', fontweight='bold')
plt.xlabel('Volume')
plt.xlim(0, counts[0] * 1.25)
plt.show()


### 4. Platform Performance: Facebook vs. Instagram
Comparing delivery volume, efficiency (CPC, CPA), and profitability (ROAS).


In [ ]:
plat_perf = ads.groupby('ad_platform').agg(
    Ads=('ad_id', 'count'),
    Spend=('ad_spend', 'sum'),
    Impressions=('impressions', 'sum'),
    Clicks=('clicks', 'sum'),
    Purchases=('purchases', 'sum'),
    Revenue=('revenue', 'sum')
).reset_index()

plat_perf['CTR (%)'] = ((plat_perf['Clicks'] / plat_perf['Impressions']) * 100).round(2)
plat_perf['CPC ($)'] = (plat_perf['Spend'] / plat_perf['Clicks']).round(2)
plat_perf['CPA ($)'] = (plat_perf['Spend'] / plat_perf['Purchases']).round(2)
plat_perf['ROAS (x)'] = (plat_perf['Revenue'] / plat_perf['Spend']).round(2)

plat_perf


### 5. Monthly Advertising Trends
Evaluating time-series trends across May, June, July, and August 2025.


In [ ]:
monthly = daily.groupby('year_month').agg(
    Spend=('spend', 'sum'),
    Revenue=('revenue', 'sum'),
    Impressions=('impressions', 'sum'),
    Clicks=('clicks', 'sum'),
    Purchases=('purchases', 'sum')
).reset_index()

monthly['ROAS'] = (monthly['Revenue'] / monthly['Spend']).round(2)
monthly['CTR (%)'] = ((monthly['Clicks'] / monthly['Impressions']) * 100).round(2)

monthly


### 6. Creative Format Performance
Analyzing which creative type delivers the highest return: Video, Stories, Carousel, or Image.


In [ ]:
format_df = ads.groupby('ad_type').agg(
    Ads=('ad_id', 'count'),
    Spend=('ad_spend', 'sum'),
    Impressions=('impressions', 'sum'),
    Clicks=('clicks', 'sum'),
    Purchases=('purchases', 'sum'),
    Revenue=('revenue', 'sum')
).reset_index()

format_df['CTR (%)'] = ((format_df['Clicks'] / format_df['Impressions']) * 100).round(2)
format_df['CR (%)'] = ((format_df['Purchases'] / format_df['Clicks']) * 100).round(2)
format_df['CPA ($)'] = (format_df['Spend'] / format_df['Purchases']).round(2)
format_df['ROAS (x)'] = (format_df['Revenue'] / format_df['Spend']).round(2)

format_df.sort_values('ROAS (x)', ascending=False)
